# BirdCLEF+ 2026 — Submission Notebook
**5-fold EfficientNet-B0 ensemble, ONNX inference on CPU**

- Model: EfficientNet-B0 (timm) with custom head
- Ensemble: 5-fold mean-logit averaging
- Format: ONNX (quantized) for fast CPU inference
- CV Score: 0.9475 macro AUC

In [15]:
# ═══════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ═══════════════════════════════════════════════════════════════════
import os, sys, time, warnings, glob
import numpy as np
import pandas as pd
import onnxruntime as ort
import librosa

warnings.filterwarnings("ignore")

ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    BASE_DIR = "/kaggle/input/birdclef-2026"
    B0_MODEL_DIR = "submissions/base_model"
    # SEResNeXt partner ensemble — NEW DATASET
    SX_MODEL_DIR = "submissions/sed_pseudo_v4"
    TEST_DIR = "/kaggle/input/competitions/birdclef-2026/test_soundscapes"
    SAMPLE_SUB = "../data/raw/sample_submission.csv"
else:
    BASE_DIR = "data/raw"
    B0_MODEL_DIR = "experiments"
    SX_MODEL_DIR = "experiments"
    TEST_DIR = os.path.join(BASE_DIR, "test_soundscapes")
    SAMPLE_SUB = os.path.join(BASE_DIR, "sample_submission.csv")

print(f"Environment:    {'Kaggle' if ON_KAGGLE else 'Local'}")
print(f"Test dir:       {TEST_DIR}")
print(f"B0 model dir:   {B0_MODEL_DIR}")
print(f"SX model dir:   {SX_MODEL_DIR}")

# Cross-architecture blend weight: final = w * B0 + (1-w) * SEResNeXt
# 0.5 = equal weighting (recommended starting point)
ENSEMBLE_BLEND_W = 0.75

# Verify mounted datasets to catch path typos before running
if ON_KAGGLE:
    print(f"\nMounted datasets in /kaggle/input/:")
    try:
        for d in sorted(os.listdir("/kaggle/input")):
            print(f"  {d}")
    except Exception as e:
        print(f"  (could not list: {e})")


Environment:    Local
Test dir:       data/raw/test_soundscapes
B0 model dir:   experiments
SX model dir:   experiments


## Configuration
Spectrogram parameters **must match training exactly** or predictions will be garbage.

In [17]:
# ═══════════════════════════════════════════════════════════════════
# SPECTROGRAM CONFIGURATION — matches training pipeline exactly
# ═══════════════════════════════════════════════════════════════════
SAMPLE_RATE = 32000
N_MELS = 128
FMAX = 16000
HOP_LENGTH = 512
N_FFT = 2048
WINDOW_SECONDS = 5.0
WINDOW_SAMPLES = int(SAMPLE_RATE * WINDOW_SECONDS)  # 160000
NUM_CLASSES = 234
SPEC_TIME_FRAMES = 313

# Load species list from sample submission (defines column order)
sample_sub = pd.read_csv("../data/raw/sample_submission.csv")
SPECIES_LIST = [c for c in sample_sub.columns if c != "row_id"]
assert len(SPECIES_LIST) == NUM_CLASSES, (
    f"Expected {NUM_CLASSES} species, got {len(SPECIES_LIST)}"
)
print(f"Species: {NUM_CLASSES}")
print(f"Sample submission rows: {len(sample_sub)}")

Species: 234
Sample submission rows: 3


## Load ONNX Models

In [18]:
# ═══════════════════════════════════════════════════════════════════
# LOAD ONNX MODELS — two groups: B0 baseline + SEResNeXt partner
#
# Each session is stored as (session, input_name) because B0 and SEResNeXt
# may have different input tensor names ('input' vs 'spec' depending on
# export tooling). The wrapper queries each session for its actual name.
# ═══════════════════════════════════════════════════════════════════


def find_b0_models(model_dir, n_folds=5):
    """Find B0 ONNX files — preserves original discovery logic."""
    paths = []
    for fold_id in range(n_folds):
        candidates = [
            os.path.join(model_dir, f"baseline_effb0_fold{fold_id}", "best_model.onnx"),
            os.path.join(
                model_dir, f"baseline_effb0_fold{fold_id}", "best_model_quantized.onnx"
            ),
            os.path.join(model_dir, f"fold{fold_id}.onnx"),
            os.path.join(model_dir, f"fold{fold_id}_quantized.onnx"),
            os.path.join(model_dir, f"best_model_fold{fold_id}.onnx"),
            os.path.join(model_dir, f"best_model_{fold_id}.onnx"),
            os.path.join(model_dir, f"base_model_{fold_id}.onnx"),
        ]
        for c in candidates:
            if os.path.exists(c):
                paths.append(c)
                break
    if not paths:
        all_onnx = sorted(
            glob.glob(os.path.join(model_dir, "**", "*.onnx"), recursive=True)
        )
        regular = [f for f in all_onnx if "quantized" not in f]
        quantized = [f for f in all_onnx if "quantized" in f]
        paths = (regular if regular else quantized)[:n_folds]
    return paths


def find_seresnext_models(model_dir, n_folds=5):
    """Find SEResNeXt ONNX files. Tries common naming patterns."""
    paths = []
    for fold_id in range(n_folds):
        candidates = [
            os.path.join(model_dir, f"seresnext_fold{fold_id}.onnx"),
            os.path.join(model_dir, f"seresnext_fold_{fold_id}.onnx"),
            os.path.join(model_dir, f"sx_fold{fold_id}.onnx"),
            os.path.join(model_dir, f"best_model_fold{fold_id}.onnx"),
            os.path.join(
                model_dir, f"seresnext_finetune_fold{fold_id}", "model_int8_fp32.onnx"
            ),
            os.path.join(
                model_dir, f"seresnext_finetune_fold{fold_id}", "best_model.onnx"
            ),
        ]
        for c in candidates:
            if os.path.exists(c):
                paths.append(c)
                break
    if not paths:
        all_onnx = sorted(
            glob.glob(os.path.join(model_dir, "**", "*.onnx"), recursive=True)
        )
        # Avoid grabbing the B0 files if both dirs happen to point at the same place
        sx_only = [
            p
            for p in all_onnx
            if "seresnext" in p.lower() or "sx" in os.path.basename(p).lower()
        ]
        paths = (sx_only if sx_only else all_onnx)[:n_folds]
    return paths


def load_sessions(paths, label):
    """Load ONNX sessions and capture input tensor name for each."""
    items = []
    for p in paths:
        sess = ort.InferenceSession(p, providers=["CPUExecutionProvider"])
        inp_name = sess.get_inputs()[0].name
        size_mb = os.path.getsize(p) / 1024 / 1024
        print(
            f'  [{label}] {os.path.basename(p)}: {size_mb:.1f} MB, input="{inp_name}"'
        )
        items.append((sess, inp_name))
    return items


# --- B0 baseline ensemble ---
print(f"B0 model search in: {B0_MODEL_DIR}")
b0_paths = find_b0_models(B0_MODEL_DIR)
print(f"Found {len(b0_paths)} B0 ONNX models:")
b0_items = load_sessions(b0_paths, "B0")

# --- SEResNeXt partner ensemble ---
print(f"\nSX model search in: {SX_MODEL_DIR}")
sx_paths = find_seresnext_models(SX_MODEL_DIR)
print(f"Found {len(sx_paths)} SEResNeXt ONNX models:")
sx_items = load_sessions(sx_paths, "SX")

# Sanity guards
if len(b0_items) == 0:
    raise RuntimeError(
        f"No B0 ONNX models found under {B0_MODEL_DIR!r}. "
        f"Cannot proceed — B0 is required as the baseline. "
        f"Verify the dataset is added to the notebook."
    )

if len(sx_items) == 0:
    print(f"\n⚠ WARNING: No SEResNeXt models found under {SX_MODEL_DIR!r}.")
    print(f"  Falling back to B0-only ensemble (your previous baseline behavior).")
    print(f"  Verify SX_MODEL_DIR path if you intended to use SEResNeXt.")
    USE_SX = False
else:
    USE_SX = True

N_B0 = len(b0_items)
N_SX = len(sx_items)
print(f"\nEnsemble: {N_B0} B0 + {N_SX} SEResNeXt models")
if USE_SX:
    print(f"Cross-architecture blend: B0={ENSEMBLE_BLEND_W}, SX={1 - ENSEMBLE_BLEND_W}")


B0 model search in: experiments
Found 0 B0 ONNX models:

SX model search in: experiments
Found 0 SEResNeXt ONNX models:


RuntimeError: No B0 ONNX models found under 'experiments'. Cannot proceed — B0 is required as the baseline. Verify the dataset is added to the notebook.

## Inference Pipeline

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# INFERENCE FUNCTIONS — two-architecture ensemble with 5× TTA
#
# Aggregation order:
#   1. For each TTA offset, mean logits within each architecture group
#      (5 folds → 1 logit vector per group per offset)
#   2. Across offsets, mean logits per architecture group
#   3. Sigmoid each group separately to get probability vectors
#   4. Linear blend: final = w * b0_probs + (1-w) * sx_probs
#
# This separates fold variance from time-shift variance from
# architecture differences, and avoids letting one arch's quirks
# bleed into the other through a single big logit pool.
#
# If SEResNeXt isn't loaded, falls through cleanly to B0-only path.
# ═══════════════════════════════════════════════════════════════════

# 5 offsets evenly spaced over the 5s window
TTA_OFFSETS_SAMPLES = [
    -int(0.50 * WINDOW_SAMPLES),  # -2.50 s
    -int(0.25 * WINDOW_SAMPLES),  # -1.25 s
     0,                           #  center
    +int(0.25 * WINDOW_SAMPLES),  # +1.25 s
    +int(0.50 * WINDOW_SAMPLES),  # +2.50 s
]

# ─── Frame-level inference: per-session pool/max blend ─────────────
# 0.0 = pool-only (legacy behaviour), 1.0 = max-only
ALPHA = 0.5


def _run_logits(sess, inp_name, x, alpha=ALPHA):
    """Read both heads from a dual-output SED ONNX and blend in logit space.
    Falls back transparently to single-output sessions (non-SED, or folds
    that haven't been re-exported yet)."""
    out_names = [o.name for o in sess.get_outputs()]
    if "logits_pool" in out_names and "logits_max" in out_names:
        pool, mx = sess.run(["logits_pool", "logits_max"], {inp_name: x})
        return ((1.0 - alpha) * pool + alpha * mx)[0]
    return sess.run(None, {inp_name: x})[0][0]

def compute_melspec(waveform):
    """Log-mel spectrogram matching training preprocessing."""
    S = librosa.feature.melspectrogram(
        y=waveform, sr=SAMPLE_RATE,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmax=FMAX,
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    if S_db.shape[1] >= SPEC_TIME_FRAMES:
        S_db = S_db[:, :SPEC_TIME_FRAMES]
    else:
        S_db = np.pad(
            S_db, ((0, 0), (0, SPEC_TIME_FRAMES - S_db.shape[1])),
            mode='constant', constant_values=S_db.min(),
        )
    return S_db


def extract_shifted_segment(waveform, center_start, offset_samples):
    """5s segment shifted by offset; zero-pad at boundaries."""
    total_samples = len(waveform)
    start = center_start + offset_samples
    end = start + WINDOW_SAMPLES

    pad_left  = max(0, -start)
    pad_right = max(0, end - total_samples)
    valid_start = max(0, start)
    valid_end   = min(total_samples, end)
    segment = waveform[valid_start:valid_end]

    if pad_left > 0 or pad_right > 0:
        segment = np.pad(segment, (pad_left, pad_right), mode='constant', constant_values=0.0)

    if len(segment) < WINDOW_SAMPLES:
        segment = np.pad(segment, (0, WINDOW_SAMPLES - len(segment)), mode='constant')
    elif len(segment) > WINDOW_SAMPLES:
        segment = segment[:WINDOW_SAMPLES]
    return segment


def _select_offsets(n_tta):
    """Pick TTA subset, always preserving symmetric spread first."""
    if n_tta >= 5:
        return TTA_OFFSETS_SAMPLES
    if n_tta == 4:
        return [TTA_OFFSETS_SAMPLES[0], TTA_OFFSETS_SAMPLES[1],
                TTA_OFFSETS_SAMPLES[3], TTA_OFFSETS_SAMPLES[4]]
    if n_tta == 3:
        return [TTA_OFFSETS_SAMPLES[0], TTA_OFFSETS_SAMPLES[2], TTA_OFFSETS_SAMPLES[4]]
    if n_tta == 2:
        return [TTA_OFFSETS_SAMPLES[2], TTA_OFFSETS_SAMPLES[4]]
    return [TTA_OFFSETS_SAMPLES[2]]



def predict_window_ensemble(
    b0_items,
    sx_items,
    waveform,
    center_start,
    n_tta=5,
    blend_w=ENSEMBLE_BLEND_W,
    use_sx=True,
):
    """
    Two-architecture ensemble inference for a single 5s window.

    Args:
        b0_items: list of (session, input_name) for B0 models
        sx_items: list of (session, input_name) for SEResNeXt models
        waveform: full soundscape audio
        center_start: window start sample index
        n_tta: 1..5 (number of TTA offsets)
        blend_w: weight on B0 in cross-arch blend
        use_sx: if False, returns B0-only probs

    Returns:
        probs: numpy array of shape (NUM_CLASSES,)
    """
    offsets = _select_offsets(n_tta)
    b0_per_offset = []
    sx_per_offset = []

    for offset in offsets:
        segment = extract_shifted_segment(waveform, center_start, offset)
        spec = compute_melspec(segment)
        x = spec[np.newaxis, np.newaxis, :, :].astype(np.float32)

        b0_fold_logits = []
        for sess, inp_name in b0_items:
            logits = _run_logits(sess, inp_name, x)
            b0_fold_logits.append(logits)
        b0_per_offset.append(np.mean(b0_fold_logits, axis=0))

        if use_sx:
            sx_fold_logits = []
            for sess, inp_name in sx_items:
                logits = _run_logits(sess, inp_name, x)
                sx_fold_logits.append(logits)
            sx_per_offset.append(np.mean(sx_fold_logits, axis=0))

    # Mean across offsets, then sigmoid
    b0_logits = np.mean(b0_per_offset, axis=0)
    b0_probs = 1.0 / (1.0 + np.exp(-b0_logits))

    if not use_sx or not sx_per_offset:
        return b0_probs

    sx_logits = np.mean(sx_per_offset, axis=0)
    sx_probs = 1.0 / (1.0 + np.exp(-sx_logits))

    return blend_w * b0_probs + (1.0 - blend_w) * sx_probs


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# PROCESS ALL TEST SOUNDSCAPES — 5× TTA + two-arch ensemble + budget fallback
# ═══════════════════════════════════════════════════════════════════

TIME_BUDGET_SECONDS = 80 * 60  # hard ceiling
TIME_WARN_SECONDS = 75 * 60  # warn at this point
N_TTA_INITIAL = 5

audio_extensions = ("*.ogg", "*.wav", "*.flac", "*.mp3")
audio_files = []
for ext in audio_extensions:
    audio_files.extend(glob.glob(os.path.join(TEST_DIR, ext)))
audio_files = sorted(audio_files)
print(f"Test soundscapes: {len(audio_files)}")

total_start = time.time()
rows = []
timing = {"audio": 0, "spec_plus_infer": 0, "windows": 0}
n_tta_current = N_TTA_INITIAL
tta_downgrades = []

for file_idx, audio_path in enumerate(audio_files):
    soundscape_id = os.path.splitext(os.path.basename(audio_path))[0]

    # Load audio
    t0 = time.perf_counter()
    y, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    timing["audio"] += time.perf_counter() - t0

    total_samples = len(y)
    n_windows = int(np.ceil(total_samples / WINDOW_SAMPLES))

    # Predict each window
    for win_idx in range(n_windows):
        center_start = win_idx * WINDOW_SAMPLES
        end_time_seconds = (win_idx + 1) * int(WINDOW_SECONDS)
        row_id = f"{soundscape_id}_{end_time_seconds}"

        t0 = time.perf_counter()
        probs = predict_window_ensemble(
            b0_items,
            sx_items,
            y,
            center_start,
            n_tta=n_tta_current,
            blend_w=ENSEMBLE_BLEND_W,
            use_sx=USE_SX,
        )
        timing["spec_plus_infer"] += time.perf_counter() - t0
        timing["windows"] += 1

        row = {"row_id": row_id}
        for sp_idx, sp in enumerate(SPECIES_LIST):
            row[sp] = float(probs[sp_idx])
        rows.append(row)

    # Time-budget check
    elapsed = time.time() - total_start
    rate = (file_idx + 1) / elapsed
    remaining = (len(audio_files) - file_idx - 1) / rate if rate > 0 else 0
    est_total = elapsed + remaining

    if est_total > TIME_BUDGET_SECONDS and n_tta_current > 1:
        old = n_tta_current
        n_tta_current = max(1, n_tta_current - 1)
        tta_downgrades.append(
            {
                "soundscape_idx": file_idx + 1,
                "elapsed_min": elapsed / 60,
                "est_total_min": est_total / 60,
                "from_tta": old,
                "to_tta": n_tta_current,
            }
        )
        print(
            f"  ⚠ TIME-BUDGET FALLBACK at soundscape {file_idx + 1}/{len(audio_files)}: "
            f"est_total={est_total / 60:.1f}min > budget={TIME_BUDGET_SECONDS / 60:.0f}min. "
            f"Dropping TTA: {old}× → {n_tta_current}×"
        )

    if (file_idx + 1) % 10 == 0 or file_idx == 0 or file_idx == len(audio_files) - 1:
        warn = "  ⚠" if est_total > TIME_WARN_SECONDS else " "
        print(
            f"{warn} [{file_idx + 1:>4}/{len(audio_files)}] "
            f"{soundscape_id}: {n_windows} windows | "
            f"TTA={n_tta_current}× | "
            f"Elapsed: {elapsed / 60:.1f}min | "
            f"ETA: {remaining / 60:.1f}min | "
            f"Total est: {est_total / 60:.1f}min"
        )

total_elapsed = time.time() - total_start
n_win = timing["windows"]
print(
    f"\nDone! {len(audio_files)} soundscapes, {n_win} windows in "
    f"{total_elapsed:.1f}s ({total_elapsed / 60:.1f}min)"
)
if n_win > 0:
    print(
        f"  Per window: audio={timing['audio'] / max(len(audio_files), 1) * 1000:.1f}ms/file, "
        f"spec+infer={timing['spec_plus_infer'] / n_win * 1000:.1f}ms/window"
    )

if tta_downgrades:
    print(f"\n⚠ TTA was downgraded {len(tta_downgrades)} time(s):")
    for d in tta_downgrades:
        print(
            f"  At soundscape {d['soundscape_idx']}: "
            f"TTA {d['from_tta']}× → {d['to_tta']}× "
            f"(elapsed {d['elapsed_min']:.1f}min, projected {d['est_total_min']:.1f}min)"
        )
else:
    print(f"\n✓ Completed with full {N_TTA_INITIAL}× TTA throughout.")


## Build and save submission

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BUILD SUBMISSION
# ═══════════════════════════════════════════════════════════════════
expected_cols = list(sample_sub.columns)

if rows:
    submission = pd.DataFrame(rows)
    # Ensure exact column match with sample submission
    for col in expected_cols:
        if col not in submission.columns:
            submission[col] = 0.0
    submission = submission[expected_cols]
else:
    # No test files found — use sample_submission as skeleton
    submission = sample_sub.copy()
    print("⚠ No audio files found — using sample submission as placeholder")

# Validate
assert list(submission.columns) == expected_cols, "Column mismatch!"
print(f"Submission shape: {submission.shape}")
print(f"Expected shape:   ({len(sample_sub)}, {len(expected_cols)})")
print(f"Columns match:    ✓")

# Quick sanity check
pred_vals = submission[SPECIES_LIST].values
print(f"\nPrediction stats:")
print(f"  Mean: {pred_vals.mean():.6f}")
print(f"  Std:  {pred_vals.std():.6f}")
print(f"  Min:  {pred_vals.min():.6f}")
print(f"  Max:  {pred_vals.max():.6f}")

submission.head()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SAVE
# ═══════════════════════════════════════════════════════════════════
submission.to_csv("submission.csv", index=False)
print(f"  {submission.shape[0]} rows × {submission.shape[1]} columns")
pd.read_csv("/kaggle/working/submission.csv")